# 07 — MinIO → regressie-analyse

Een end-to-end mini-analyse, **direct vanuit MinIO** (zonder Trino/SQL):

1. Twee bronze Delta-tabellen lezen via `delta-rs`.
2. JSON-payloads ontleden naar tabulaire kolommen.
3. Joinen op BSN, leeftijd en dienstverband-duur afleiden.
4. Lineaire regressie (sklearn) — bruto jaarloon als functie van leeftijd & tenure.
5. Scatter + regressie-lijn plotten.

**Spoiler:** de synthetische dataset is bewust ongecorreleerd. R² zal
rond 0 liggen — dat is een _eerlijke_ uitkomst die laat zien hoe een
regressie eruit ziet als er geen signaal in de data zit. De workflow
(lezen → joinen → fitten → visualiseren) is hetzelfde voor echte data.

In [ ]:
import os, json, warnings

# Wire delta-rs's trust store to the platform CA bundle BEFORE importing
# uwv_lab.s3 — object_store-rs reads SSL_CERT_FILE only at first use, and
# older kernel images pre-date the auto-wiring in the helper.
_ca = os.environ.get('UWV_CA_BUNDLE', '/etc/uwv-ca/ca.crt')
if os.path.isfile(_ca):
    os.environ.setdefault('SSL_CERT_FILE', _ca)
    os.environ.setdefault('AWS_CA_BUNDLE', _ca)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from uwv_lab import s3, env

warnings.filterwarnings('ignore', message='Unverified HTTPS request')
env()

## 1. Tabellen laden

`persona_created` (10k personen, demografie) en `polisadm_ikv`
(15k dienstverbanden, met bruto jaarloon) komen uit `uwv-bronze` —
Delta-tabellen geschreven door de bronze-streaming-job.

In [ ]:
personas_raw  = s3.read_delta('s3a://uwv-bronze/uwv/persona_created/')
polisadm_raw  = s3.read_delta('s3a://uwv-bronze/uwv/polisadm_ikv/')
print('personas:', personas_raw.shape)
print('polisadm:', polisadm_raw.shape)
personas_raw.head(2)

## 2. JSON-payload uitpakken

Bronze-tabellen bewaren de oorspronkelijke Kafka-payload als één string —
robuust voor schema-evolutie maar onhandig om mee te rekenen. Een
korte helper plat de `payload`-kolom in een gewone DataFrame.

In [ ]:
def explode_payload(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(json.loads(p)['payload'] for p in df['payload'])

personas = explode_payload(personas_raw)
polisadm = explode_payload(polisadm_raw)
print('personas:', personas.shape, '— cols:', list(personas.columns))
print('polisadm:', polisadm.shape, '— cols:', list(polisadm.columns))

## 3. Features afleiden

- **leeftijd** uit `geboortedatum` (gemeten op een vaste referentie-datum).
- **tenure_years** uit `aanvang_dienstverband`.
- Filter onwaarschijnlijke uitschieters: leeftijd ∈ [18, 67], salaris > 0.

In [ ]:
REF_DATE = pd.Timestamp('2026-01-01')
personas['age'] = ((REF_DATE - pd.to_datetime(personas['geboortedatum'])).dt.days / 365.25).astype(int)
polisadm['tenure_years'] = ((REF_DATE - pd.to_datetime(polisadm['aanvang_dienstverband'])).dt.days / 365.25)

merged = (
    polisadm[['bsn', 'loon_bruto_jaar', 'tenure_years']]
    .merge(personas[['bsn', 'age', 'geslacht']], on='bsn', how='inner')
)
merged = merged[(merged['age'].between(18, 67)) & (merged['loon_bruto_jaar'] > 0) & (merged['tenure_years'] > 0)]
print('rows kept:', len(merged))
merged.head(3)

In [ ]:
merged[['age', 'tenure_years', 'loon_bruto_jaar']].describe().round(1)

## 4. Eerst eenvoudig — salaris ~ leeftijd

Eén feature (`age`) tegen één target (`loon_bruto_jaar`). Reken het lineaire
verband en kijk hoeveel variantie het verklaart (R²).

In [ ]:
X = merged[['age']].values
y = merged['loon_bruto_jaar'].values

mdl_simple = LinearRegression().fit(X, y)
y_hat = mdl_simple.predict(X)
r2 = r2_score(y, y_hat)

print(f'salaris = {mdl_simple.intercept_:.0f} + {mdl_simple.coef_[0]:.1f} * leeftijd')
print(f'R² = {r2:.3f}   (≈ 0 → geen lineair signaal in deze synthetische data)')

In [ ]:
# Scatter + regressie-lijn
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(merged['age'], merged['loon_bruto_jaar'], alpha=0.05, s=10, color='#F37726')
xs = np.linspace(18, 67, 100).reshape(-1, 1)
ax.plot(xs, mdl_simple.predict(xs), color='black', linewidth=2, label=f'OLS (R² = {r2:.2f})')
ax.set_xlabel('Leeftijd (jaar)')
ax.set_ylabel('Bruto jaarloon (EUR)')
ax.set_title('Bruto jaarloon vs. leeftijd — synthetische UWV-bronze data')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()

## 5. Multivariate — leeftijd + tenure

Zelfde target, twee features. Kijk of de twee samen iets toevoegen.

In [ ]:
X2 = merged[['age', 'tenure_years']].values
mdl_multi = LinearRegression().fit(X2, y)
r2_multi = r2_score(y, mdl_multi.predict(X2))

print(f'salaris = {mdl_multi.intercept_:.0f}'
      f' + {mdl_multi.coef_[0]:.1f} * leeftijd'
      f' + {mdl_multi.coef_[1]:.1f} * tenure')
print(f'R² = {r2_multi:.3f}')

## 6. Conclusie

Op deze synthetische bronze-data is er **geen lineair verband** tussen
demografie/tenure en het bruto jaarloon — coëfficiënten dicht bij nul,
R² ≈ 0. Eerlijke uitkomst: het generator-script trekt salaris uniform
willekeurig uit een vaste range, los van de andere velden.

Op echte data zou je hier in plaats van stoppen:
- non-lineaire features toevoegen (`age²`, log-loon),
- categorische features one-hot encoden (`geslacht`, `regio_code`),
- splitsen in train/test, en
- model-keuze verbreden (Lasso, Ridge, gradient-boosting).

Het *interessante* hier is dat we de hele pipeline — Delta uit MinIO,
join, feature-engineering, fit, visualisatie — in één notebook
binnen het platform hebben gedraaid, met je Keycloak-identiteit als
audit-trail.